In [0]:
df = spark.read.table ("gizmobox.bronze.py_payments")
display(df)

In [0]:
from pyspark.sql.functions import *


In [0]:
df_casted = (
    df
    .select('payments_id', 
    date_format('payment_timestamp',"yyyy-MM-dd").cast('date').alias('payment_date'),
    date_format('payment_timestamp',"HH:mm:ss").cast('timestamp').alias('payment_time'),
    'payment_status', 
    'payment_method')
)

display(df_casted)

In [0]:
from pyspark.sql.functions import when
df_status = (
    df_casted.select("payments_id",
                     "payment_date",
                     "payment_time",
                     "payment_date",
                     when(df_casted.payment_status == 1, "Success")
                     .when(df_casted.payment_status == 2, "Pending")
                     .when(df_casted.payment_status == 3, "Cancelled")
                     .when(df_casted.payment_status == 4, "Failed").alias("payment_status" ),
                     "payment_method")
)
display(df_status)

In [0]:
# Drop duplicate payment_date column before writing
df_final = df_status.select("payments_id", "payment_date", "payment_time", "payment_status", "payment_method")
df_final.write.mode("overwrite").saveAsTable("gizmobox.silver.py_payments")